## Exercise 5

### Part 1

In [ ]:
import numpy as np
from scipy.stats import t

# Number of samples
n = 100

# Generate samples from U(0,1)
u = np.random.uniform(0,1,n)

# Transform samples
y = np.exp(u)

# Estimate integral
theta_hat = np.mean(y)

# Standard deviation of the sample
s = np.std(y, ddof=1)

# Estimate standard error
se = s / np.sqrt(n)

# Find 95% confidence interval using the estimate of the integral after using 100 samples
# t critical value
t_crit = t.ppf(0.975, df=n-1)

# Confidence interval
lower_ci = theta_hat - t_crit * se
upper_ci = theta_hat + t_crit * se

print(f"Monte Carlo Estimate = {theta_hat:.6f}")
print(f"95% CI = ({lower_ci:.6f}, {upper_ci:.6f})")


Monte Carlo Estimate = 1.733047
95% CI = (1.645361, 1.820733)


### Part 2

In [ ]:
# 50 uniforms -> 100 function evaluations, since each antithetic sample uses two function evaluations
n = 50

# Generate random variables
u = np.random.uniform(0, 1, n)

# Antithetic observations
y = (np.exp(u) + np.exp(1 - u)) / 2

# Compute estimate
estimate = np.mean(y)

# Standard deviation and standard error
s = np.std(y, ddof=1)
se = s / np.sqrt(n)

# 95% confidence interval 
t_crit = t.ppf(0.975, df=n-1)

lower = estimate - t_crit * se
upper = estimate + t_crit * se

print("Antithetic estimate:", estimate)
print("95% CI:", (lower, upper))


Antithetic estimate: 1.7085343006270273
95% CI: (1.6916715397569946, 1.72539706149706)


### Part 3

In [ ]:
# Number of samples
n = 100

# Generate random variables and transform to function
U = np.random.uniform(0, 1, n)

# Define Xi and Zi
X = np.exp(U)
Z = U
mu_Z = 0.5

# Estimate optimal c
c = -np.cov(X, Z, ddof=1)[0, 1] / np.var(Z, ddof=1)

# Construct Yi
Y = X + c * (Z - mu_Z)

# Compute estimate
theta_hat = np.mean(Y)

# Standard deviation and standard error
s = np.std(Y, ddof=1)
se = s / np.sqrt(n)

# 95% confidence interval
t_crit = t.ppf(0.975, df=n-1)

lower_ci = theta_hat - t_crit * se
upper_ci = theta_hat + t_crit * se

print("Control variate estimate:", theta_hat)
print("Optimal c:", c)
print("95% CI:", (lower_ci, upper_ci))

Control variate estimate: 1.7126414846744422
Optimal c: -1.6685820131916946
95% CI: (1.700863128443861, 1.7244198409050235)


### Part 4

In [ ]:
# Set parameters
m = 10   # Strata
n = 100   # Samples per stratum

estimates = []

# stratified sampling
for i in range(m):
    a = i / m
    b = (i + 1) / m

    U = np.random.uniform(a, b, n)
    X = np.exp(U)

    estimates.append(np.mean(X))

estimates = np.array(estimates)

# Estimator
theta_hat = np.mean(estimates)

# Variance + SE
s = np.std(estimates, ddof=1)
se = s / np.sqrt(m)

# 95% confidence interval
t_crit = t.ppf(0.975, df=m-1)

lower = theta_hat - t_crit * se
upper = theta_hat + t_crit * se

print("Stratified estimate:", theta_hat)
print("95% CI:", (lower, upper))

Stratified estimate: 1.7191053575638293
95% CI: (1.349384615398405, 2.088826099729254)


### Part 5

In [ ]:
import random
import heapq
import statistics
import math

# Set parameters
M = 10
mean_service = 8
mean_interarrival = 1.0

num_customers = 10000
num_runs = 10


def sim_run():

    current_time = 0.0

    arrivals = 0
    blocked = 0
    departures = 0
    busy_servers = 0

    event_list = []

    first_arrival = random.expovariate(1 / mean_interarrival)
    heapq.heappush(event_list, (first_arrival, "arrival"))

    while arrivals < num_customers:

        event_time, event_type = heapq.heappop(event_list)
        current_time = event_time

        if event_type == "arrival":

            arrivals += 1

            next_arrival = current_time + random.expovariate(1 / mean_interarrival)
            heapq.heappush(event_list, (next_arrival, "arrival"))

            if busy_servers < M:
                busy_servers += 1

                service_time = random.expovariate(1 / mean_service)
                departure_time = current_time + service_time

                heapq.heappush(event_list, (departure_time, "departure"))

            else:
                blocked += 1

        elif event_type == "departure":
            busy_servers -= 1
            departures += 1

    return blocked / arrivals, departures


# Run simulations
Y = []
X = []

for _ in range(num_runs):
    y, x = sim_run()
    Y.append(y)
    X.append(x)

Y_bar = statistics.mean(Y)
X_bar = statistics.mean(X)

# Covariance and variance
cov_YX = sum((Y[i] - Y_bar) * (X[i] - X_bar) for i in range(num_runs)) / (num_runs - 1)
var_X = sum((X[i] - X_bar) ** 2 for i in range(num_runs)) / (num_runs - 1)

c = cov_YX / var_X

# Control variate estimator
Y_cv = [
    Y[i] - c * (X[i] - X_bar)
    for i in range(num_runs)
]

# Statistics
mean_cv = statistics.mean(Y_cv)
std_cv = statistics.stdev(Y_cv)

# 95% confidence interval with 9 degrees of freedom
t_value = 2.262
margin = t_value * std_cv / math.sqrt(num_runs)

lower = mean_cv - margin
upper = mean_cv + margin


# Results
print("Original blocking probabilities:")
for r in Y:
    print(f"{r:.5f}")

print("\nControl variate results:")
print(f"Mean = {mean_cv:.5f}")
print(f"95% CI = [{lower:.5f}, {upper:.5f}]")

Original blocking probabilities:
0.12680
0.12720
0.12230
0.12010
0.12450
0.12240
0.12220
0.12010
0.12610
0.11800

Control variate results:
Mean = 0.12297
95% CI = [0.12280, 0.12314]


### Part 7

In [ ]:
# Function - crude Monte Carlo
def crude_mc(a, n):
    Z = np.random.normal(0, 1, n)
    return np.mean(Z > a)

# Function - importance sampling
def importance_sampling(a, n, sigma=1.0):
    Y = np.random.normal(a, sigma, n)

    # Indicator
    indicator = (Y > a)

    # Likelihood ratio:
    weights = np.exp(-a * Y + (a**2) / 2)

    return np.mean(indicator * weights)


# Run experiments
np.random.seed(42)

ns = [1000, 10000, 50000]
as_values = [2, 4]

for a in as_values:
    print("\n============================")
    print(f"a = {a}")
    print("============================")

    true_val = 1 - 0.5 * (1 + np.math.erf(a / np.sqrt(2)))
    print("True value (approx):", true_val)

    for n in ns:
        mc = crude_mc(a, n)
        is_est = importance_sampling(a, n, sigma=1.0)

        print(f"\nn = {n}")
        print("Crude MC:", mc)
        print("Importance Sampling:", is_est)


a = 2
True value (approx): 0.022750131948179098

n = 1000
Crude MC: 0.024
Importance Sampling: 0.02299192775645415

n = 10000
Crude MC: 0.0221
Importance Sampling: 0.023208759610248327

n = 50000
Crude MC: 0.02244
Importance Sampling: 0.022829011045719612

a = 4
True value (approx): 3.167124183311998e-05

n = 1000
Crude MC: 0.0
Importance Sampling: 3.137696088829966e-05

n = 10000
Crude MC: 0.0
Importance Sampling: 3.0391272305327644e-05

n = 50000
Crude MC: 0.0
Importance Sampling: 3.174710057039718e-05


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_46124\2268257085.py:32: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  true_val = 1 - 0.5 * (1 + np.math.erf(a / np.sqrt(2)))


### Part 8

In [ ]:
import numpy as np

# Part 8

# Importance sampling estimator
def estimate_theta(n, lam):
    X = np.random.exponential(1/lam, n)

    # Importance weights:
    # Y = e^X / (lambda e^{-lambda X}) * 1[X<=1]
    Y = (1/lam) * np.exp((1 + lam) * X) * (X <= 1)

    return np.mean(Y)


# Estimate variance for given lambda
def estimate_variance(n, lam, R=200):
    estimates = np.zeros(R)

    for r in range(R):
        estimates[r] = estimate_theta(n, lam)

    return np.var(estimates, ddof=1)


# Search for optimal lambda
lambdas = np.linspace(0.2, 3.0, 30)

n = 5000   # Samples per run
R = 200    # Repetitions for variance estimation

variances = []

for lam in lambdas:
    var_lam = estimate_variance(n, lam, R)
    variances.append(var_lam)
    print(f"lambda={lam:.2f}, variance={var_lam:.6f}")

variances = np.array(variances)

# Optimal lambda
best_index = np.argmin(variances)
best_lambda = lambdas[best_index]

print("\n====================")
print("Optimal lambda:", best_lambda)
print("Min variance:", variances[best_index])
print("====================")

lambda=0.20, variance=0.002574
lambda=0.30, variance=0.002667
lambda=0.39, variance=0.001394
lambda=0.49, variance=0.001067
lambda=0.59, variance=0.001085
lambda=0.68, variance=0.000890
lambda=0.78, variance=0.000719
lambda=0.88, variance=0.000737
lambda=0.97, variance=0.000630
lambda=1.07, variance=0.000666
lambda=1.17, variance=0.000538
lambda=1.26, variance=0.000546
lambda=1.36, variance=0.000477
lambda=1.46, variance=0.000686
lambda=1.55, variance=0.000641
lambda=1.65, variance=0.000653
lambda=1.74, variance=0.000791
lambda=1.84, variance=0.000703
lambda=1.94, variance=0.000691
lambda=2.03, variance=0.000687
lambda=2.13, variance=0.000883
lambda=2.23, variance=0.000939
lambda=2.32, variance=0.000747
lambda=2.42, variance=0.000966
lambda=2.52, variance=0.001066
lambda=2.61, variance=0.001108
lambda=2.71, variance=0.000943
lambda=2.81, variance=0.001152
lambda=2.90, variance=0.001328
lambda=3.00, variance=0.001380

Optimal lambda: 1.3586206896551725
Min variance: 0.000477402923529814

### Part 9

In [ ]:
# Parameters
xm = 1.0
alpha = 3.0
n = 10000

# Sample from g(x) = Pareto(xm, alpha-1)
U = np.random.uniform(0, 1, n)
X = xm * (1 - U) ** (-1 / (alpha - 1))

# Target density f(x)
def f(x):
    return alpha * xm**alpha / x**(alpha + 1)

# Proposal density g(x)
def g(x):
    return (alpha - 1) * xm**(alpha - 1) / x**alpha

# Importance sampling estimator
weights = f(X) / g(X)

theta_hat = np.mean(X * weights)

print("IS estimate (Pareto mean):", theta_hat)
print("True mean:", alpha * xm / (alpha - 1))

IS estimate (Pareto mean): 1.5
True mean: 1.5
